# Alternative Gap Patching via Daily Archives (SOLUSDT)

During previous data validation, missing minutes were detected in the `SOLUSDT` series. To test whether high-resolution daily dumps contain the missing records (which can occasionally be omitted during Binance monthly aggregation), we will:

1. Re-download the full monthly dataset from scratch.
2. Compile a baseline Parquet file and locate exact calendar dates containing missing timestamps.
3. Dynamically download `daily` archives solely for those impacted dates.
4. Merge both sources (`UNION`) and apply forward-filling (`ffill`) only on timestamps corresponding to actual exchange halts.

> **Note:** The existing `SOLUSDT_2021_2026.parquet` file has been intentionally removed to ensure a clean end-to-end run.

In [1]:
import os
import sys
import shutil
import urllib.request
import zipfile
from pathlib import Path
from datetime import datetime
import duckdb

# Ensure working directory is always the project root
if Path(os.getcwd()).name == "notebooks":
  os.chdir("..")

# Add project root to sys.path
sys.path.append(os.getcwd())

from src.data_pipeline import download_and_extract, get_months_list, process_and_validate, DTYPES_DICT

# --- Configuration ---
SYMBOL = 'SOLUSDT'
START_DATE = datetime(2021, 7, 1)
END_DATE = datetime(2026, 6, 1)
INTERVAL = '1m'

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Remove existing target parquet if present
parquet_path = PROCESSED_DIR / f"{SYMBOL}_2021_2026.parquet"
if parquet_path.exists():
  parquet_path.unlink()
  print(f"Removed existing {parquet_path.name}")

In [2]:
# Step 1: Download all monthly archives for SOLUSDT
print("Fetching monthly archives...")
months = get_months_list(START_DATE, END_DATE)

# Download and extract raw monthly CSVs
download_and_extract(SYMBOL, months)

Fetching monthly archives...

--- Downloading data for SOLUSDT ---


Fetching SOLUSDT:  47%|████▋     | 28/60 [00:58<01:00,  1.89s/it]


Attempt 1/3 failed for SOLUSDT-1m-2023-11.zip: The read operation timed out


Fetching SOLUSDT:  97%|█████████▋| 58/60 [02:35<00:04,  2.10s/it]


Attempt 1/3 failed for SOLUSDT-1m-2026-05.zip: The read operation timed out


Fetching SOLUSDT: 100%|██████████| 60/60 [03:13<00:00,  3.22s/it]


In [3]:
# Step 2: Compile raw monthly CSVs into an unpatched base Parquet file
conn = duckdb.connect()
symbol_dir = RAW_DIR / SYMBOL

print("Compiling raw monthly CSVs to base Parquet...")
conn.execute(f"""
  COPY (
    SELECT 
      epoch_ms(column00) AS open_time,
      column01 AS open, column02 AS high, column03 AS low, column04 AS close,
      column05 AS volume, column07 AS qav, column08 AS n_trades,
      column09 AS buy_volume, column10 AS buy_qav
    FROM read_csv('{symbol_dir}/*.csv', header=False, columns={DTYPES_DICT}, ignore_errors=true)
    ORDER BY open_time
  ) TO '{parquet_path}' (FORMAT PARQUET)
""")
print(f"Base Parquet created at {parquet_path}")

Compiling raw monthly CSVs to base Parquet...
Base Parquet created at data\processed\SOLUSDT_2021_2026.parquet


In [4]:
# Step 3: Identify specific dates with missing timestamps
print("Scanning for missing dates across expected 1m grid...")
missing_dates_query = f"""
  WITH expected_index AS (
    SELECT unnest(generate_series(
      TIMESTAMP '2021-07-01 00:00:00', 
      TIMESTAMP '2026-07-01 00:00:00' - INTERVAL 1 MINUTE, 
      INTERVAL 1 MINUTE
    )) AS ts
  ),
  missing_minutes AS (
    SELECT ts FROM expected_index 
    EXCEPT 
    SELECT open_time FROM read_parquet('{parquet_path}')
  )
  SELECT DISTINCT strftime(ts, '%Y-%m-%d') AS missing_date
  FROM missing_minutes
  ORDER BY missing_date
"""

missing_dates = [row[0] for row in conn.execute(missing_dates_query).fetchall()]
print(f"Found {len(missing_dates)} dates with missing timestamps:")
print(missing_dates)

Scanning for missing dates across expected 1m grid...
Found 5 dates with missing timestamps:
['2022-02-26', '2022-02-27', '2022-02-28', '2022-04-01', '2022-04-02']


In [5]:
# Step 4: Download daily archives for the identified dates
DAILY_BASE_URL = "https://data.binance.vision/data/futures/um/daily/klines"
PATCH_DIR = RAW_DIR / f"{SYMBOL}_patch"
PATCH_DIR.mkdir(parents=True, exist_ok=True)

print("Fetching daily patch archives...")
for date in missing_dates:
  zip_name = f"{SYMBOL}-{INTERVAL}-{date}.zip"
  url = f"{DAILY_BASE_URL}/{SYMBOL}/{INTERVAL}/{zip_name}"
  zip_path = PATCH_DIR / zip_name
  
  try:
    if not (PATCH_DIR / f"{SYMBOL}-{INTERVAL}-{date}.csv").exists():
      urllib.request.urlretrieve(url, zip_path)
      with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(PATCH_DIR)
      zip_path.unlink()
      print(f"Successfully downloaded daily patch: {date}")
  except Exception as e:
    print(f"Failed to fetch {date} from daily archive (likely exchange halt): {e}")

Fetching daily patch archives...
Successfully downloaded daily patch: 2022-02-26
Successfully downloaded daily patch: 2022-02-27
Successfully downloaded daily patch: 2022-02-28
Successfully downloaded daily patch: 2022-04-01
Successfully downloaded daily patch: 2022-04-02


In [6]:
# Step 5: Merge monthly & daily data, apply fallback ffill, and clean up
print("Merging datasets and applying forward-fill fallback...")

conn.execute(f"""
  COPY (
    WITH expected_index AS (
      SELECT unnest(generate_series(
        TIMESTAMP '2021-07-01 00:00:00', 
        TIMESTAMP '2026-07-01 00:00:00' - INTERVAL 1 MINUTE, 
        INTERVAL 1 MINUTE
      )) AS open_time
    ),
    daily_patches AS (
      SELECT 
        epoch_ms(column00) AS open_time, column01 AS open, 
        column02 AS high, column03 AS low, column04 AS close, 
        column05 AS volume, column07 AS qav, column08 AS n_trades, 
        column09 AS buy_volume, column10 AS buy_qav
      FROM read_csv('{PATCH_DIR}/*.csv', header=False, columns={DTYPES_DICT}, ignore_errors=true)
    ),
    combined_raw AS (
      SELECT * FROM read_parquet('{parquet_path}')
      UNION
      SELECT * FROM daily_patches
    ),
    joined AS (
      SELECT 
        e.open_time,
        c.open, c.high, c.low, c.close,
        COALESCE(c.volume, 0) AS volume,
        COALESCE(c.qav, 0) AS qav,
        COALESCE(c.n_trades, 0) AS n_trades,
        COALESCE(c.buy_volume, 0) AS buy_volume,
        COALESCE(c.buy_qav, 0) AS buy_qav
      FROM expected_index e
      LEFT JOIN combined_raw c ON e.open_time = c.open_time
    ),
    filled AS (
      SELECT *,
           last_value(close IGNORE NULLS) OVER (ORDER BY open_time) AS ffill_close
      FROM joined
    )
    SELECT 
      open_time,
      COALESCE(open, ffill_close) AS open,
      COALESCE(high, ffill_close) AS high,
      COALESCE(low, ffill_close) AS low,
      COALESCE(close, ffill_close) AS close,
      volume, qav, n_trades, buy_volume, buy_qav
    FROM filled
    ORDER BY open_time
  ) TO '{parquet_path}.temp' (FORMAT PARQUET)
""")

# Replace base parquet with the patched & verified file
Path(f"{parquet_path}.temp").replace(parquet_path)
conn.close()

# Cleanup raw directory
shutil.rmtree(symbol_dir, ignore_errors=True)
shutil.rmtree(PATCH_DIR, ignore_errors=True)

print(f"Patched dataset saved to {parquet_path.name}. Raw temporary files removed.")

Merging datasets and applying forward-fill fallback...
Patched dataset saved to SOLUSDT_2021_2026.parquet. Raw temporary files removed.


In [7]:
process_and_validate(SYMBOL)


--- Processing & Validating SOLUSDT ---
Running integrity checks...
All checks passed successfully!
Cleaning up raw CSV files...
Clean up failed
